In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [48]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device, EpochTrainer, build_seq, get_yf_data
from operator import itemgetter
from sklearn.preprocessing import RobustScaler as Scaler

# Parameters

In [49]:
model_output_filename = "lstm_checkpoint"

## model config

In [50]:
batch_size = 60
epochs = 100
hidden_size = 64
num_output = 1
num_layers = 2
dropout = 0.2
bidirectional = False
patience = 5
test_size = 0.3
lr = 0.001

In [51]:
seq_len = 20
data_len = 500


eval_method = 'RMSE'
criterion = nn.MSELoss()

In [52]:
input_cols = ["Open", "High", "Low", "Close", "Volume"]
identifier_cols = ['Date']
target_cols = ['Close']

# Get Sample Data

## Dataset additional Parameters

In [53]:
steps_ahead = 5
ticker = "VOO"
start = "2015-01-01"
end = "2026-04-29"

## Load data from yfinance

In [54]:
data = get_yf_data( ticker , start , end)

[*********************100%***********************]  1 of 1 completed


# Data Processing

## Scale inputs and target

In [55]:
x_scaler = Scaler()
y_scaler = Scaler()

X_all = x_scaler.fit_transform(data[input_cols].values)
y_all = y_scaler.fit_transform(data[target_cols].values)

## Build Sequence

In [56]:
id_all = data[identifier_cols].values.flatten().tolist()

Seqs = build_seq( seq_len , X_all, steps_ahead, id_all , y_all  )

X_seq, y_seq, X_id, y_id = itemgetter('X_Seq', 'y_Seq', 'X_id', 'y_id')(Seqs)

In [57]:
X_seq, y_seq, X_id, y_id = X_seq[-data_len:], y_seq[-data_len:], X_id[-data_len:], y_id[-data_len:]

## Convert to Tensor

In [58]:
X_tensor = torch.from_numpy(np.asarray(X_seq, dtype=np.float32))
y_tensor = torch.tensor(np.array(y_seq), dtype=torch.float32)

In [59]:
full_dataset = TensorDataset(X_tensor, y_tensor)

## Split data into training and testing

In [60]:
input_size = X_tensor.shape[-1]

In [61]:
test_records = int(test_size * len(full_dataset))
train_records = len(full_dataset) - test_records

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_records, test_records],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Prepare Model

## Prepare config of the model

In [62]:
preprocess_config = {
    "seq_length": seq_len,
    "input_size": input_size,
    "task_type": "classification",
}

In [63]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
}

## Get Model

In [64]:
device = get_device()

In [65]:
model = LSTM_Model(
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    num_layers=num_layers,
    dropout=dropout,
    bidirectional=bidirectional,
).to(device)

In [66]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

## Early stopping and checkpoint setup

In [67]:
early_stopping = EarlyStopping(
    patience=patience,
    path=f"../checkpoints/{model_output_filename}.pt",
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "target_cols": target_cols,
        "input_cols": input_cols,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "steps_ahead" : steps_ahead,
    },
)

# Loop epochs and train model

In [68]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = eval_method
)

In [69]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch   1 | Train Loss: 2.2413 | Val Loss: 1.8722 | RMSE: 1.3683
Epoch   2 | Train Loss: 1.3779 | Val Loss: 0.7036 | RMSE: 0.8388
Epoch   3 | Train Loss: 0.2893 | Val Loss: 0.1230 | RMSE: 0.3507
Epoch   4 | Train Loss: 0.2044 | Val Loss: 0.1030 | RMSE: 0.3209
Epoch   5 | Train Loss: 0.0795 | Val Loss: 0.0637 | RMSE: 0.2524
Epoch   6 | Train Loss: 0.0975 | Val Loss: 0.0976 | RMSE: 0.3124
Epoch   7 | Train Loss: 0.0960 | Val Loss: 0.0617 | RMSE: 0.2484
Epoch   8 | Train Loss: 0.0744 | Val Loss: 0.0489 | RMSE: 0.2211
Epoch   9 | Train Loss: 0.0760 | Val Loss: 0.0477 | RMSE: 0.2184
Epoch  10 | Train Loss: 0.0675 | Val Loss: 0.0451 | RMSE: 0.2123
Epoch  11 | Train Loss: 0.0658 | Val Loss: 0.0470 | RMSE: 0.2168
Epoch  12 | Train Loss: 0.0630 | Val Loss: 0.0395 | RMSE: 0.1987
Epoch  13 | Train Loss: 0.0567 | Val Loss: 0.0358 | RMSE: 0.1891
Epoch  14 | Train Loss: 0.0545 | Val Loss: 0.0350 | RMSE: 0.1871
Epoch  15 | Train Loss: 0.0511 | Val Loss: 0.0343 | RMSE: 0.1852
Epoch  16 | Train Loss: 0